# Fungi — Google Colab

Segment fungi microscopy videos with the full Fungi web UI in Colab.

**Before you start:** Runtime → Change runtime type → **GPU** (recommended).

Run all cells in order. The first run downloads SAM2 weights (~900 MB) and may take 10–20 minutes.

In [ ]:
# Configuration
REPO_URL = "https://github.com/standard-model-lagrangian/fungi.git"
REPO_BRANCH = "main"
WORK_DIR = "/content/fungi"

# Optional: DeepCell token for CellSAM (https://users.deepcell.org)
DEEPCELL_TOKEN = ""  # e.g. "your-token-here"

In [ ]:
import os
import subprocess
from pathlib import Path

work_dir = Path(WORK_DIR)
if work_dir.exists():
    print(f"Using existing clone at {work_dir}")
else:
    subprocess.check_call(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(work_dir)])

os.chdir(work_dir)
print("Repository ready at", work_dir)

In [ ]:
# Install Node.js (needed to build the React frontend)
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
!apt-get install -y nodejs
!node --version
!npm --version

In [ ]:
# Build the frontend (uses relative /api URLs for Colab)
!cd frontend && npm ci && npm run build
print("Frontend built to frontend/dist/")

In [ ]:
# Install Python dependencies (Colab already includes PyTorch + CUDA)
!pip install -q -r colab/requirements-colab.txt

if DEEPCELL_TOKEN:
    os.environ["DEEPCELL_ACCESS_TOKEN"] = DEEPCELL_TOKEN
    print("DeepCell token set.")
else:
    print("No DeepCell token — CellSAM may fall back to threshold segmentation.")

In [ ]:
# Pre-load backend dependencies (SAM2 clone, weights download, CellSAM install)
# This can take several minutes on first run.
import sys
sys.path.insert(0, str(work_dir / "backend"))
os.chdir(work_dir / "backend")

print("Installing / verifying ML dependencies (first run is slow)...")
import sam2_pipeline  # noqa: F401
print("Backend dependencies ready.")

In [ ]:
import threading
import time

PORT = 8000
server_error = []

def run_server():
    try:
        os.chdir(work_dir)
        os.environ["FUNGI_COLAB"] = "1"
        os.environ["PORT"] = str(PORT)
        subprocess.run([sys.executable, "colab/serve_colab.py"], check=True)
    except Exception as exc:
        server_error.append(exc)

thread = threading.Thread(target=run_server, daemon=True)
thread.start()

for _ in range(60):
    time.sleep(1)
    import urllib.request
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/api/segmentation-presets", timeout=2)
        print(f"Server is up on port {PORT}")
        break
    except Exception:
        pass
else:
    raise RuntimeError(f"Server did not start: {server_error}")

In [ ]:
from google.colab import output

print("Opening Fungi in a Colab window...")
output.serve_kernel_port_as_window(PORT, path="/", height=800)